In [1]:
# 9.1 Ejercicio 1: Mayoría con N=5

N = 5
partition_A = [0, 1]
partition_B = [2, 3, 4]

# Cálculo de mayoría
majority = (N // 2) + 1

print("="*50)
print("EJERCICIO 1: Mayoría con N=5")
print("="*50)
print(f"N = {N}")
print(f"Mayoría = floor(N/2) + 1 = floor({N}/2) + 1 = {majority}")
print(f"\nPartición A = {partition_A}, tamaño = {len(partition_A)}")
print(f"Partición B = {partition_B}, tamaño = {len(partition_B)}")
print(f"\n¿Partición A tiene mayoría? {len(partition_A) >= majority}")
print(f"¿Partición B tiene mayoría? {len(partition_B) >= majority}")
print(f"\nCONCLUSIÓN: La mayoría está en {partition_B}")
print("="*50)

EJERCICIO 1: Mayoría con N=5
N = 5
Mayoría = floor(N/2) + 1 = floor(5/2) + 1 = 3

Partición A = [0, 1], tamaño = 2
Partición B = [2, 3, 4], tamaño = 3

¿Partición A tiene mayoría? False
¿Partición B tiene mayoría? True

CONCLUSIÓN: La mayoría está en [2, 3, 4]


In [2]:
# 9.2 Ejercicio 2: Quórum W=3, R=3 con partición

N = 5
W = 3
R = 3
partition_A = [0, 1]
partition_B = [2, 3, 4]

print("="*50)
print("EJERCICIO 2: Quórum W=3, R=3 durante partición")
print("="*50)
print(f"Configuración: N={N}, W={W}, R={R}")
print(f"Partición A = {partition_A}, tamaño = {len(partition_A)}")
print(f"Partición B = {partition_B}, tamaño = {len(partition_B)}")

print("\n--- Disponibilidad en Partición A ---")
print(f"Puede ESCRIBIR? {len(partition_A) >= W} (necesita W={W}, tiene {len(partition_A)})")
print(f"Puede LEER? {len(partition_A) >= R} (necesita R={R}, tiene {len(partition_A)})")

print("\n--- Disponibilidad en Partición B ---")
print(f"Puede ESCRIBIR? {len(partition_B) >= W} (necesita W={W}, tiene {len(partition_B)})")
print(f"Puede LEER? {len(partition_B) >= R} (necesita R={R}, tiene {len(partition_B)})")

print("\nCONCLUSIÓN:")
print(f"- Lado A: NO puede operar (sin quórum)")
print(f"- Lado B: SÍ puede operar (tiene quórum)")
print(f"- R+W={R+W} > N={N} ⇒ Consistencia fuerte (CP)")
print("="*50)

EJERCICIO 2: Quórum W=3, R=3 durante partición
Configuración: N=5, W=3, R=3
Partición A = [0, 1], tamaño = 2
Partición B = [2, 3, 4], tamaño = 3

--- Disponibilidad en Partición A ---
Puede ESCRIBIR? False (necesita W=3, tiene 2)
Puede LEER? False (necesita R=3, tiene 2)

--- Disponibilidad en Partición B ---
Puede ESCRIBIR? True (necesita W=3, tiene 3)
Puede LEER? True (necesita R=3, tiene 3)

CONCLUSIÓN:
- Lado A: NO puede operar (sin quórum)
- Lado B: SÍ puede operar (tiene quórum)
- R+W=6 > N=5 ⇒ Consistencia fuerte (CP)


### 9.2 Ejercicio 2: Quórum W=3, R=3 durante partición

**¿Qué pasa durante la partición?**

Con N=5, W=3, R=3, tanto lecturas como escrituras necesitan contactar al menos 3 nodos.

- **Partición A {0,1}**: 2 nodos → NO puede completar lecturas ni escrituras (< quórum)
- **Partición B {2,3,4}**: 3 nodos → SÍ puede completar lecturas y escrituras (alcanza quórum)

**Por qué:** Como R+W=6 > N=5, los quórums se solapan, garantizando consistencia fuerte. El sistema sacrifica disponibilidad en el lado minoritario.

In [3]:
# 9.3 Ejercicio 3: Dos escrituras concurrentes (LWW)

class SimpleLWW:
    def __init__(self):
        self.data = {}
        self.clock = 0
    def write(self, key, value, side):
        self.clock += 1
        ts = self.clock
        self.data[key] = {'value': value, 'ts': ts, 'side': side}
        return ts
    def get(self, key):
        return self.data.get(key)

print("="*50)
print("EJERCICIO 3: Escrituras concurrentes + LWW")
print("="*50)
store_A, store_B = SimpleLWW(), SimpleLWW()
print("\n--- ANTES: Escrituras concurrentes ---")
ts_A = store_A.write('clave', 'ValorA', 'LadoA')
ts_B = store_B.write('clave', 'ValorB', 'LadoB')
print(f"LadoA: {store_A.get('clave')}, ts={ts_A}")
print(f"LadoB: {store_B.get('clave')}, ts={ts_B}")
print("\n--- DESPUÉS: Reconciliación LWW ---")
val_A, val_B = store_A.get('clave'), store_B.get('clave')
winner = val_A if val_A['ts'] > val_B['ts'] else val_B
print(f"Comparando: ts_A={val_A['ts']}, ts_B={val_B['ts']}")
print(f"GANADOR (LWW): {winner['value']} con ts={winner['ts']} de {winner['side']}")
print(f"\nCriterio LWW: timestamp lógico mayor = última escritura.")
print("="*50)

EJERCICIO 3: Escrituras concurrentes + LWW

--- ANTES: Escrituras concurrentes ---
LadoA: {'value': 'ValorA', 'ts': 1, 'side': 'LadoA'}, ts=1
LadoB: {'value': 'ValorB', 'ts': 1, 'side': 'LadoB'}, ts=1

--- DESPUÉS: Reconciliación LWW ---
Comparando: ts_A=1, ts_B=1
GANADOR (LWW): ValorB con ts=1 de LadoB

Criterio LWW: timestamp lógico mayor = última escritura.


### 9.3 Ejercicio 3: Escrituras concurrentes + LWW

**¿Cómo gana una en LWW?** El algoritmo Last-Write-Wins elige el valor con **timestamp lógico mayor**.

**Mecanismo:** Cada escritura obtiene un timestamp incremental. Tras curar la partición, se comparan todos los timestamps y el mayor se propaga como valor final.

**Evidencia:** En la salida, LadoB tiene ts=1, que gana sobre LadoA. El sistema converge al valor con timestamp más alto.

---

### 9.4 Ejercicio 4: CP vs AP en Big Data

**CP (Configuración crítica):** Para configuración de clúster Kafka, parámetros de Spark, feature flags de modelos ML. Leer config vieja puede causar ejecución con parámetros erróneos, incompatibilidades entre servicios, o uso de modelos obsoletos.
**AP (Logs/Métricas):** Para logs centralizados, métricas de Prometheus, observabilidad. Es crítico NO perder eventos aunque haya desfase temporal. Con CP podríamos perder logs de errores durante particiones,
**Consecuencias:** Usar AP en config crítica = fallos en cascada. Usar CP en observabilidad = huecos irrecuperables en auditoría.